From the project root run  
pip install -e .

In [9]:
#To get the datasets for a disease run the following
# disases = ["CM", "PCD", "PH", "PCD", "ASD", "nonTarget"]
from ExTSP.tripletSets_exTSP import get_tripletSets_with_exTSP, variantCountsWithDuplicates, subsample_tripletSets
# All CM variante
df_target_all = get_tripletSets_with_exTSP("CM")
# CM P variant
df_target_p = get_tripletSets_with_exTSP("CM", "P")
total, duplication_count =variantCountsWithDuplicates(df_target_p)
print(f"CM P variants: {total}, {duplication_count.max()}")
# CM B variant
numVars = df_target_p.Variant.nunique()
numVars = None
df_target_b = get_tripletSets_with_exTSP("CM", "B", numVars=numVars)
total, duplication_count =variantCountsWithDuplicates(df_target_b)
print(f"CM B variants: {total}, {duplication_count.max()}")
# CM PLP variant    
# df_target_plp = get_tripletSets_with_exTSP("CM", "PLP")
# print(f"CM PLP variants: {variantCountsWithDuplicates(df_target_plp)}")
# # CM BLB variant
# df_target_blb = get_tripletSets_with_exTSP("CM", "BLB", numVars=df_target_plp.Variant.nunique())
# print(f"CM BLB variants: {variantCountsWithDuplicates(df_target_blb)}")
# # nonTarget PLP variant
# df_nonTarget_plp = get_tripletSets_with_exTSP("nonTarget", "PLP")
# # nonTarget BLB variant
# df_nonTarget_blb = get_tripletSets_with_exTSP("nonTarget", "BLB")
# # ASD cases gtex
# df_ASD_cases = get_tripletSets_with_exTSP("ASD", "case")
# #ASD controls gtex
# df_ASD_controls = get_tripletSets_with_exTSP("ASD", "control")
# #ASD cases brainspan
# df_ASD_cases_brainspan = get_tripletSets_with_exTSP("ASD", "case", brainspan=True)
# #ASD controls brainspan
# df_ASD_controls_brainspan = get_tripletSets_with_exTSP("ASD", "control", brainspan=True)












CM P variants: 398.0, 1.0
CM B variants: 111.0, 1.0


In [13]:
import importlib
import ExTSP.isoformSelection.IsoformSelection_Variant as iso_module
importlib.reload(iso_module)
from ExTSP.isoformSelection.IsoformSelection_Variant import exTSP_selected_isoform, exTSP_selected_isoform_Tissue
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
Tissue = "Heart_Left_Ventricle"
selectionColumn = "exTSP"
df_p_isoform = exTSP_selected_isoform(df_target_p, selectionColumn=selectionColumn)
df_b_isoform = exTSP_selected_isoform(df_target_b, selectionColumn=selectionColumn)
df_p_isoform_tissue = df_p_isoform[df_p_isoform.Tissue == Tissue]
df_b_isoform_tissue = df_b_isoform[df_b_isoform.Tissue == Tissue]
df_p_isoform_tissue = exTSP_selected_isoform_Tissue(df_target_p, Tissue)
df_b_isoform_tissue = exTSP_selected_isoform_Tissue(df_target_b, Tissue)


def auc(pos,neg):
    y_true = [1]*len(pos) + [0]*len(neg)
    y_pred = pos + neg
    return roc_auc_score(y_true, y_pred)

# Pathogenic versus benign in the picked tissue
auc_isoPath = auc(df_p_isoform_tissue.IsoPath.tolist(), df_b_isoform_tissue.IsoPath.tolist())
auc_exTSP = auc(df_p_isoform_tissue.exTSP.tolist(), df_b_isoform_tissue.exTSP.tolist())
print(f"Pathogenic versus benign in the picked tissue AUC of isoPath: {auc_isoPath}")
print(f"Pathogenic versus benign in the picked tissue AUC of exTSP: {auc_exTSP}")


# Pathogenic versus benign in the picked tissue

pos = df_p_isoform[df_p_isoform.Tissue == Tissue].IsoPath.to_list()
neg = df_b_isoform.IsoPath.to_list() + df_p_isoform[df_p_isoform.Tissue != Tissue].IsoPath.to_list()
auc_isoPath = auc(pos, neg)
print(f"Pathogenic versus benign in the picked tissue AUC of isoPath: {auc_isoPath}")

pos = df_p_isoform[df_p_isoform.Tissue == Tissue].exTSP.to_list()
neg = df_b_isoform.exTSP.to_list() + df_p_isoform[df_p_isoform.Tissue != Tissue].exTSP.to_list()
auc_exTSP = auc(pos, neg)
print(f"Pathogenic versus benign in the picked tissue AUC of exTSP: {auc_exTSP}")











Pathogenic versus benign in the picked tissue AUC of isoPath: 0.9431278011680022
Pathogenic versus benign in the picked tissue AUC of exTSP: 0.9104078953325184
Pathogenic versus benign in the picked tissue AUC of isoPath: 0.5968548149261845
Pathogenic versus benign in the picked tissue AUC of exTSP: 0.7672652194613265


In [6]:
pos = df_p_isoform[df_p_isoform.Tissue == Tissue].IsoPath.to_list()
neg = df_b_isoform.IsoPath.to_list() + df_p_isoform[df_p_isoform.Tissue != Tissue].IsoPath.to_list()
auc_isoPath = auc(pos, neg)
print(f"Pathogenic versus benign in the picked tissue AUC of isoPath: {auc_isoPath}")

pos = df_p_isoform[df_p_isoform.Tissue == Tissue].exTSP.to_list()
neg = df_b_isoform.exTSP.to_list() + df_p_isoform[df_p_isoform.Tissue != Tissue].exTSP.to_list()
auc_exTSP = auc(pos, neg)
print(f"Pathogenic versus benign in the picked tissue AUC of exTSP: {auc_exTSP}")












Pathogenic versus benign in the picked tissue AUC of isoPath: 0.6950460699849936
Pathogenic versus benign in the picked tissue AUC of exTSP: 0.7300831559367802


In [7]:
df_p_isoform.columns

Index(['Gene', 'Transcript', 'Transcript_id', 'Variant', 'MutPred2', 'IsoPath',
       'ptse', 'exTSP', 'meanTPM', 'Tissue', 'MANE_Select',
       'MANE_Plus_Clinical', 'ClinVar_annotation'],
      dtype='object')

In [ ]:


y_true = [1]*len(pos_iso) + [0]*len(neg_iso)
y_pred = pos_iso + neg_iso

print(roc_auc_score(y_true, y_pred))


0.6763249395276945


In [3]:
from ExTSP.tripletSets_exTSP import get_tripletSets_with_exTSP
df_ASD_cases_brainspan = get_tripletSets_with_exTSP("ASD", "case", brainspan=True)

In [15]:
int(len(neg_iso_p)/len(neg_iso_b))

2